# 🐇 Light Rabbit — pannello del nodo White Rabbit su KR260

Carica il firmware **v15**, porta la scheda in **`TRACK_PHASE`** contro il WR-ZEN,
e mostra **vUART**, **frequenzimetro** e stato del lock.

### ⚠️ Prima di eseguire
| | |
|---|---|
| **Fibra** | **SFP BiDi verso il WR-ZEN** (non loopback) |
| **Kernel** | dev'essere **root** (serve `/dev/mem` + `xmutil`) — jupyter va lanciato con `sudo` |
| **vUART** | **un solo consumatore alla volta**: se hai uno script/monitor attivo da shell, chiudilo o le celle qui daranno output vuoto |
| **AXI** | non leggere `0xA00xxxxx` **prima** di aver caricato l'app (cella 2) → blocca la scheda |

Eseguire le celle **in ordine** la prima volta.

> **Riprendere una sessione** (kernel nuovo, firmware gia' caricato e nodo agganciato):
> esegui la **cella 1** e poi vai dove vuoi — le celle si agganciano all'AXI da sole.
> **NON rieseguire la cella 2**: ricaricherebbe la PL e butterebbe via il lock.

## 1 — Helper: mmap, vUART, frequenzimetro

In [ ]:
import mmap, os, ctypes, time, subprocess, re
from datetime import datetime

WB_BASE, WB_SPAN = 0xA0000000, 0x10000     # bridge Wishbone: vUART + WRPC
FM_BASE, FM_SPAN = 0xA0030000, 0x1000      # freq_counter (TXUSRCLK2)
TDR, RDR, MAGIC  = 0x510, 0x514, 0x000     # registri vUART / magic 'WRPC'
FM_GATE, FM_CNT, FM_STAT = 0x00, 0x04, 0x08

assert os.geteuid() == 0, "Il kernel NON gira come root: rilancia jupyter con sudo"

_mem = os.open('/dev/mem', os.O_RDWR | os.O_SYNC)

def _map(base, span):
    return mmap.mmap(_mem, span, mmap.MAP_SHARED,
                     mmap.PROT_READ | mmap.PROT_WRITE, offset=base)

_wb = _fm = None                      # mappati DOPO il load (cella 2)

def rd(m, off):      return ctypes.c_uint32.from_buffer(m, off).value
def wr(m, off, val): ctypes.c_uint32.from_buffer(m, off).value = val & 0xffffffff

def attach():
    """Mappa i registri PL. Da chiamare SOLO dopo xmutil loadapp."""
    global _wb, _fm
    _wb, _fm = _map(WB_BASE, WB_SPAN), _map(FM_BASE, FM_SPAN)
    m = rd(_wb, MAGIC)
    if m != 0x57525043:               # 'WRPC'
        raise RuntimeError(f"MAGIC=0x{m:08X} != 'WRPC' → app WR non caricata")
    print(f"✅ bridge WB agganciato (MAGIC='WRPC') @ 0x{WB_BASE:08X}")

def ensure_attached():
    """Aggancia l'AXI se non lo e' gia'. Serve quando si riprende una sessione senza
    rieseguire la cella 2 (che ricaricherebbe la PL e butterebbe via il lock)."""
    if _wb is None:
        attach()

# ---------- vUART: parla con la console del softcore WRPC (LM32) ----------
def _drain(quiet=0.30, hard=3.0):
    out, t0, last = bytearray(), time.time(), time.time()
    while time.time() - t0 < hard:
        v = rd(_wb, RDR)
        if v & 0x100:                        # bit8 = byte valido
            out.append(v & 0xff); last = time.time()
        elif time.time() - last > quiet:
            break
        else:
            time.sleep(0.002)
    return out.decode('ascii', 'replace')

def wrc(cmd="", quiet=0.30, hard=3.0, echo=True):
    """Manda un comando alla vUART e restituisce l'output."""
    ensure_attached()
    _drain(0.05, 0.3)                        # svuota il pendente
    for ch in (cmd + "\r").encode('ascii', 'replace'):
        for _ in range(200000):
            if rd(_wb, TDR) & 0x100:         # TX ready
                break
        wr(_wb, TDR, ch)
    time.sleep(0.05)
    txt = _drain(quiet, hard)
    if echo:
        print(f"wrc# {cmd}\n{txt}")
    return txt

def stat():
    """'stat' è un TOGGLE dell'output continuo → ritenta finché trova la riga lnk:."""
    for _ in range(3):
        t = wrc("stat", echo=False)
        if "lnk:" in t:
            return t
        time.sleep(0.7)
    return t

def parse_stat(txt):
    """Estrae i campi k:v dell'ultima riga lnk: (la più fresca)."""
    lines = [l for l in txt.splitlines() if "lnk:" in l]
    if not lines:
        return {}
    d = dict(re.findall(r"(\w+):('[^']*'|-?\w+)", lines[-1]))
    return {k: v.strip("'") for k, v in d.items()}

# ---------- frequenzimetro ----------
def freq_meas(gate_cycles=125_000_000, pl_clk_hz=125_000_000):
    """Misura TXUSRCLK2. Ritorna (Hz, conteggio grezzo, secondi di gate)."""
    ensure_attached()
    wr(_fm, FM_GATE, gate_cycles)
    gate_s = gate_cycles / pl_clk_hz
    time.sleep(gate_s + 0.4)
    rd(_fm, FM_STAT)                          # la lettura azzera il flag
    time.sleep(gate_s + 0.4)
    cnt = rd(_fm, FM_CNT)
    return cnt / gate_s, cnt, gate_s

def sh(cmd, timeout=90):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=timeout)
    return (r.stdout + r.stderr).strip()

print("helper pronti — NON è ancora stato toccato l'AXI (si mappa nella cella 2)")

## 2 — Carica il firmware v15 sulla PL

`xmutil unloadapp` → `loadapp` → verifica che i **timestamp di `/dev/uio*` siano freschi**
(è la prova che la PL è stata **programmata davvero**, non un deploy finto).

👀 **Guarda la scheda:** subito dopo il load **LED2** (cage SFP) deve **respirare** = fabric vivo.

In [ ]:
APP = "kr260_wr_sdm_app_v15"

print(sh("xmutil unloadapp") or "(niente da scaricare)")
time.sleep(1)
print(sh(f"xmutil loadapp {APP}"))
time.sleep(3)

print("\n--- /dev/uio (uio4/5/6 = app WR; devono avere l'ora DI ADESSO) ---")
print(sh("for u in /sys/class/uio/uio*; do echo \"  $(basename $u): $(cat $u/name)\"; done"))
print(sh("ls -la --time-style=+%H:%M:%S /dev/uio* | awk '{print \"  \"$6\"  \"$7}'"))
print(f"\n  adesso sono le {datetime.now():%H:%M:%S}")

attach()          # ora è lecito toccare l'AXI

## 3 — Laser SFP + livelli ottici

⚠️ **Trappola:** ogni riprogrammazione della PL **azzera il GPO** che abilita il laser →
va riacceso **dopo ogni load**, altrimenti il TX resta spento e il link non sale mai.

Atteso: **RX ≈ −6 dBm** (se è LOS/garbage: fibra staccata o SFP rotto → fermarsi qui).

In [ ]:
print(sh("python3 /home/ubuntu/tx_enable.py on"))
print()
ddm = sh("python3 /home/ubuntu/read_sfp_ddm.py")
print("\n".join(l for l in ddm.splitlines()
                if re.search(r"power|Temperatura|bias|LOS|TX_DISABLE|fault", l, re.I)))

## 4 — vUART: la console del softcore WRPC

La `wrc()` parla direttamente col softcore LM32 dentro la PL, via il bridge Wishbone.
Se il softcore non fosse partito, qui non uscirebbe **nulla** (vUART muta = wrc non boota).

In [ ]:
time.sleep(6)                 # lascia bootare il wrc dopo il load
wrc("")                       # prompt: deve rispondere 'wrc#'
wrc("help", hard=4.0);

## 5 — Bring-up White Rabbit: `slave` + PTP

MAC fisso, **autoneg ON** (col partner reale deve completare), modo **slave**, `ptp start`.

In [ ]:
for c in ["ptp stop", "mac set 00:11:22:33:44:55", "mdio an 1", "mode slave", "ptp start"]:
    wrc(c); time.sleep(1)
print("→ bring-up lanciato; il lock di fase arriva tipicamente in pochi secondi")

## 6 — Frequenzimetro: la QPLL0 sta davvero girando alla frequenza giusta?

Conta i fronti di **TXUSRCLK2** (il clock che esce dalla QPLL0 sterzata via SDM) su una
finestra di gate. **Atteso ≈ 62.5 MHz.**

- `62 500 000` → QPLL0 in bolla ✅
- `63 476 562` → FBDIV a 65 invece di 66 (da correggere via DRP)
- `0` → clock morto

⚠️ Niente `devmem2` qui: su aarch64 fa accessi a 64 bit e va in SIGBUS sugli offset
`addr%8==4` (ci abbiamo perso due giorni). Questa cella usa mmap a **32 bit**.

In [ ]:
hz, cnt, gate_s = freq_meas()
print(f"gate            = {gate_s:.3f} s")
print(f"FREQ_CNT        = {cnt:,}".replace(",", " "))
print(f"→ TXUSRCLK2     = {hz/1e6:.4f} MHz")

err_ppm = (hz - 62.5e6) / 62.5e6 * 1e6
if abs(err_ppm) < 500:
    print(f"\n✅ QPLL0 SANA — scarto {err_ppm:+.1f} ppm dai 62.5 MHz nominali")
elif cnt == 0:
    print("\n❌ conteggio 0 → clock morto (TX in reset? QPLL0 non locka?)")
else:
    print(f"\n⚠️ fuori tolleranza: {err_ppm:+.0f} ppm — controllare FBDIV/SDM")

## 7 — Monitor del lock: si arriva a `TRACK_PHASE`?

Legge in loop `stat` + `pll stat` e disegna **`cko`** (clock offset dal master, in ps) e
**`md`** (uscita del DAC che sterza l'SDM).

**Obiettivo:** `MFL1` **`MPL1`**, `ss = TRACK_PHASE`, **`cko` entro pochi ps**.
Sulla scheda: **LED1 fisso** (cage) e **UF1 che lampeggia 1/s** (PPS).

In [ ]:
# NB: la scheda ha matplotlib 3.5.1 (sistema) ma matplotlib_inline 0.2.2 (~/.local),
# che pretende RcParams._get → il backend "inline" di Jupyter esplode.
# Aggiriamo: rendering con Agg + figure mostrate come PNG. Nessuna dipendenza nuova.
import sys, io
sys.modules.pop("matplotlib.pyplot", None)
import matplotlib
matplotlib.use("Agg", force=True)
import matplotlib.pyplot as plt
from IPython.display import clear_output, display, Image

def show(fig):
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=90, bbox_inches="tight")
    plt.close(fig)
    display(Image(data=buf.getvalue()))

N = 25                       # letture (~10 s l'una)
t, cko_h, md_h = [], [], []

for i in range(N):
    s  = parse_stat(stat())
    ps = wrc("pll stat", echo=False)
    mfl = "MFL1" if "MFL1" in ps else "MFL0"
    mpl = "MPL1" if "MPL1" in ps else "MPL0"

    if s:
        t.append(i)
        cko_h.append(int(s.get("cko", 0)))
        md_h.append(int(s.get("md", 0)))

    clear_output(wait=True)
    ok = (mpl == "MPL1" and s.get("ss") == "TRACK_PHASE")
    print(f"[{i+1:2d}/{N}] {datetime.now():%H:%M:%S}   {'🏆 PHASE LOCK' if ok else '⏳ in aggancio…'}")
    print(f"   lnk:{s.get('lnk','?')}  lock:{s.get('lock','?')}  ptp:{s.get('ptp','?')}  "
          f"ss:{s.get('ss','?')}")
    print(f"   {mfl} {mpl}   cko:{s.get('cko','?')} ps   md:{s.get('md','?')}   "
          f"setp:{s.get('setp','?')}")

    if len(t) > 1:
        fig, ax = plt.subplots(1, 2, figsize=(11, 3.2))
        ax[0].plot(t, cko_h, marker="o", ms=3)
        ax[0].axhline(0, lw=.8, color="#888")
        ax[0].set_title("cko — clock offset dal master [ps]"); ax[0].set_xlabel("lettura")
        ax[0].grid(alpha=.3)
        ax[1].plot(t, md_h, marker="o", ms=3, color="#D55E00")
        ax[1].set_title("md — uscita DAC (steering SDM)"); ax[1].set_xlabel("lettura")
        ax[1].grid(alpha=.3)
        fig.tight_layout()
        show(fig)

    time.sleep(8)

print("\nfine monitor.")
if cko_h:
    tail = cko_h[-10:]
    print(f"cko sulle ultime {len(tail)} letture: min {min(tail)} / max {max(tail)} ps")

## 8 — La GUI del WRPC dentro il notebook

Il firmware ha il comando **`gui`**: la classica schermata di monitor del WR PTP Core
(tempo TAI, stato PPSI, servo, parametri di timing). Qui la catturiamo dalla vUART per
**30 s**, la renderizziamo live e la **fermiamo da soli** inviando `q`.

Non è testo lineare: il `gui` pilota un terminale con **posizionamento assoluto del cursore**,
quindi la cella contiene un mini-emulatore VT100 (griglia di caratteri + colori ANSI → HTML).

**Tre trappole, tutte già gestite qui dentro:**
1. **`stat` nudo è un TOGGLE** dell'output continuo: se è acceso, il suo flusso si sovrappone
   al `gui` e lo cancella. La cella verifica che la linea sia silenziosa e, se non lo è, lo spegne.
2. **`ESC[J` ≠ `ESC[2J`**: il `gui` disegna le etichette **una sola volta** e poi ridipinge solo i
   valori usando `ESC[J` (= cancella *dal cursore in giù*). Trattarlo come clear-screen totale
   fa sparire tutte le etichette.
3. Il `gui` va avanti finché non premi un tasto → se non mandi `q` continua a sputare byte e
   **inquina ogni lettura successiva** della vUART.

In [ ]:
# import completi: la cella dev'essere autosufficiente anche se salti la 7
from IPython.display import HTML, display, clear_output

ROWS, COLS = 26, 90
# SGR ANSI → colori CSS (il gui usa 01;3x = bold + colore)
_FG = {"30":"#555","31":"#e06c75","32":"#98c379","33":"#e5c07b",
       "34":"#61afef","35":"#c678dd","36":"#56b6c2","37":"#e6e6e6","39":"#e6e6e6"}

class Screen:
    """Mini-VT100: interpreta il sottoinsieme ANSI usato dal gui del WRPC."""
    def __init__(self):
        self._wipe(); self.last = None; self.pend = b""
    def _wipe(self):
        self.buf = [[(" ", "")] * COLS for _ in range(ROWS)]
        self.buf = [[(" ", "") for _ in range(COLS)] for _ in range(ROWS)]
        self.r = self.c = 0; self.sgr = ""
    def feed(self, data):
        data = self.pend + data; self.pend = b""
        i, n = 0, len(data)
        while i < n:
            b = data[i]
            if b == 0x1b:
                if i + 1 >= n:
                    self.pend = data[i:]; return
                if data[i+1] == ord("["):
                    j = i + 2
                    while j < n and data[j] not in b"JHfKmABCD":
                        j += 1
                    if j >= n:
                        self.pend = data[i:]; return      # CSI spezzata fra due letture
                    self._csi(data[i+2:j].decode("ascii","replace"), chr(data[j]))
                    i = j + 1; continue
            ch = chr(b)
            if ch == "\n":   self.r += 1; self.c = 0
            elif ch == "\r": self.c = 0
            elif b >= 32:
                if self.r < ROWS and self.c < COLS:
                    self.buf[self.r][self.c] = (ch, self.sgr)
                self.c += 1
                if self.c >= COLS: self.c = 0; self.r += 1
            self.r = min(self.r, ROWS - 1)
            i += 1
    def _csi(self, params, final):
        if final == "J":
            # ESC[2J = schermo intero; ESC[J / ESC[0J = solo dal cursore in giù.
            # Il gui usa il secondo a ogni refresh: confonderli cancella le etichette.
            if params == "2":
                if any(ch != " " for row in self.buf for ch, _ in row):
                    self.last = self.buf
                self._wipe()
            elif params in ("", "0"):
                for c in range(self.c, COLS): self.buf[self.r][c] = (" ", "")
                for r in range(self.r + 1, ROWS):
                    self.buf[r] = [(" ", "") for _ in range(COLS)]
        elif final in "Hf":
            p = [int(x) if x.isdigit() else 1 for x in (params.split(";") + ["1","1"])[:2]]
            self.r, self.c = max(0, p[0]-1), max(0, p[1]-1)
        elif final == "K":
            for c in range(self.c, COLS): self.buf[self.r][c] = (" ", "")
        elif final == "m":
            self.sgr = params
    def html(self):
        buf = self.buf
        if not any(ch != " " for row in buf for ch, _ in row):
            buf = self.last or buf                     # frame appena azzerato
        out = []
        for row in buf:
            line, cur, run = [], None, []
            for ch, sgr in row:
                col = _FG.get(sgr.split(";")[-1], "#e6e6e6") if sgr else "#e6e6e6"
                bold = "font-weight:600;" if sgr.startswith("01") else ""
                key = (col, bold)
                if key != cur and run:
                    line.append(f'<span style="color:{cur[0]};{cur[1]}">{"".join(run)}</span>'); run = []
                cur = key; run.append(ch.replace("&","&amp;").replace("<","&lt;"))
            if run:
                line.append(f'<span style="color:{cur[0]};{cur[1]}">{"".join(run)}</span>')
            out.append("".join(line).rstrip())
        body = "\n".join(out).rstrip()
        return HTML('<pre style="background:#1e1e1e;color:#e6e6e6;padding:12px;'
                    'border-radius:6px;line-height:1.25;font-size:12.5px;'
                    f'overflow-x:auto">{body}</pre>')

def _raw(sec):
    """Lettura GREZZA della vUART: il gui manda byte di escape, non testo →
    _drain() (che decodifica in ascii) li rovinerebbe."""
    out, t0 = bytearray(), time.time()
    while time.time() - t0 < sec:
        v = rd(_wb, RDR)
        if v & 0x100: out.append(v & 0xff)
        else: time.sleep(0.002)
    return bytes(out)

def _send_raw(s):
    for ch in s.encode():
        while not (rd(_wb, TDR) & 0x100): pass
        wr(_wb, TDR, ch)

def _quiet():
    """'stat' nudo è un TOGGLE: se il flusso continuo è acceso, si mangia il gui → spegnilo."""
    if len(_raw(1.2)) > 50:
        print("… flusso 'stat' continuo acceso: lo spengo (è un toggle)")
        _send_raw("stat\r"); time.sleep(0.5); _raw(2.0)

def wrc_gui(seconds=30, fps=1.0):
    """Mostra la GUI del WRPC per N secondi, poi la chiude con 'q'."""
    ensure_attached()      # si aggancia da sola: la cella 2 (che ricarica la PL) non serve
    _quiet()
    _send_raw("gui\r")
    scr, t0 = Screen(), time.time()
    try:
        while time.time() - t0 < seconds:
            scr.feed(_raw(1.0 / fps))
            clear_output(wait=True)
            display(scr.html())
            print(f"⏱️  {int(time.time()-t0):2d}/{seconds}s — chiudo da solo con 'q'")
    finally:
        _send_raw("q")                        # SEMPRE, anche se interrompi il kernel
        time.sleep(0.4); _raw(1.5)            # svuota la coda del gui
    print("✅ gui chiusa, vUART di nuovo libera per gli altri comandi")

wrc_gui(30)

## 9 — Console libera

Cella riusabile per interrogare il nodo: `pll stat`, `stat`, `mdio dump`, `ptp`, `time`…

In [ ]:
wrc("pll stat", hard=4.0);